In [58]:
import sys
sys.path.insert(0, r"d:\AI\PythonProjects\AtomWorldBench\src")

import pandas as pd
from pathlib import Path
from utils.extract_data import extract_from_string
from utils.dataloader import load_cif_file_from_string
from evaluation.metrics import check_atom_counts, match_structures, compute_exact_match_positional_metrics

model_names = ["qwen3_4B", "qwen3_8B", "qwen3_14B", "qwen3_32B", "o3", "o4_mini", "gemini_2_5_pro", "llama3_70B"]
# model_name   = "o4_mini"
action_names = ['add_atom_action', 'insert_between_atoms_action', 'rotate_around_atom_action', 'move_atom_action', 'move_towards_atom_action']

In [59]:
def classify_wrong_type(row, use_exact_match=False):
    """
    Re-classify a row from an old wrongs CSV using the same logic as
    AtomWorldEvaluator._process_single_output.
    
    Returns the wrong_type string, or None if the row is actually correct.
    """
    generated_output = row['generated_output']
    target_cif = row['target_cif']

    # Step 1: Extract CIF from generated output
    generated_cif = extract_from_string(str(generated_output), format="cif")
    if generated_cif is None:
        return "OutputFormatError"

    # Step 2: Parse target and generated structures
    try:
        output_structure = load_cif_file_from_string(target_cif, primitive=False)
    except Exception as e:
        raise ValueError(f"Error loading target CIF: {e}")

    generated_structure = load_cif_file_from_string(generated_cif, primitive=False)
    if generated_structure is None:
        return "CIFParsingError"

    # Step 3: Check atom counts
    if not check_atom_counts(output_structure, generated_structure):
        return "AtomCountMismatch"

    # Step 4: Match structures
    if use_exact_match:
        rmsd, max_diff = compute_exact_match_positional_metrics(output_structure, generated_structure)
    else:
        rmsd, max_diff = match_structures(output_structure, generated_structure, primitive_cell=False)

    if rmsd == -1:
        return "StructureMismatch"

    # Actually correct - shouldn't be in wrongs
    return None

In [60]:
from pathlib import Path
from tqdm import tqdm

def relabel_old_wrongs_csv(csv_path, action_name=None):
    """
    Read an old-format wrongs CSV and add wrong_type labels.
    
    Args:
        csv_path: Path to the old wrongs CSV file.
        action_name: The action name (e.g. 'move_all_action') to decide metric type.
                     If None, inferred from the path.
    Returns:
        DataFrame with added 'is_error' and 'wrong_type' columns.
    """
    csv_path = Path(csv_path)
    df = pd.read_csv(csv_path)

    # Infer action name from path if not provided
    if action_name is None:
        for part in csv_path.parts:
            if part.endswith("_action"):
                action_name = part
                break

    use_exact_match = (action_name or "").lower() == "move_all_action"
    print(f"Processing: {csv_path}")
    print(f"Action: {action_name}, use_exact_match: {use_exact_match}")
    print(f"Rows: {len(df)}")

    wrong_types = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying"):
        wt = classify_wrong_type(row, use_exact_match=use_exact_match)
        wrong_types.append(wt)

    df['wrong_type'] = wrong_types
    df['is_error'] = df['wrong_type'].notna()

    # Rows where wrong_type is None are actually correct (false wrongs)
    false_wrongs = df['wrong_type'].isna().sum()
    if false_wrongs > 0:
        print(f"WARNING: {false_wrongs} rows appear to be correct (not actually wrong)")

    # Print distribution
    print("\nWrong type distribution:")
    print(df['wrong_type'].value_counts(dropna=False))
    
    return df

In [61]:
for model_name in model_names:
    for action_name in action_names:
        base_dir = Path(f"../../results/AtomWorld/{model_name}/{action_name}")
        if not base_dir.exists():
            print(f"Skipping {action_name}: directory not found")
            continue

        date_dirs = [d for d in base_dir.iterdir() if d.is_dir()]
        if not date_dirs:
            print(f"Skipping {action_name}: no date folders found")
            continue
        date_folder = max(date_dirs, key=lambda d: d.name).name
        
        old_wrong_file = base_dir / date_folder / f"{action_name}_evaluation_wrongs_fixed.csv"
        if not old_wrong_file.exists():
            print(f"Skipping {action_name}: {old_wrong_file.name} not found in {date_folder}")
            continue

        df = relabel_old_wrongs_csv(str(old_wrong_file), action_name=action_name)
        output_path = str(old_wrong_file).replace(".csv", "_relabeled.csv")
        df.to_csv(output_path, index=False)
        print(f"Saved to {output_path}\n")

Processing: ..\..\results\AtomWorld\qwen3_4B\add_atom_action\20250812_140136\add_atom_action_evaluation_wrongs_fixed.csv
Action: add_atom_action, use_exact_match: False
Rows: 110


Classifying:   1%|          | 1/110 [00:00<00:13,  7.82it/s]

d:\AI\PythonProjects\AtomWorldBench\.conda\Lib\site-packages\pymatgen\core\periodic_table.py:252: UserWarning: No Pauling electronegativity for Rf. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  if not hasattr(other, "X") or not hasattr(other, "symbol"):
d:\AI\PythonProjects\AtomWorldBench\.conda\Lib\site-packages\pymatgen\core\periodic_table.py:255: UserWarning: No Pauling electronegativity for Rf. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  x2 = float("inf") if other.X != other.X else other.X
Classifying:   5%|▌         | 6/110 [00:00<00:03, 29.63it/s]d:\AI\PythonProjects\AtomWorldBench\src\utils\dataloader.py:33: UserWarning: Issues encountered while parsing CIF: 12 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  structures = parser.parse_structures(primitive=primitive, check_occu=False)
d:\AI\P


Wrong type distribution:
wrong_type
StructureMismatch    96
OutputFormatError    13
None                  1
Name: count, dtype: int64
Saved to ..\..\results\AtomWorld\qwen3_4B\add_atom_action\20250812_140136\add_atom_action_evaluation_wrongs_fixed_relabeled.csv

Skipping insert_between_atoms_action: no date folders found
Processing: ..\..\results\AtomWorld\qwen3_4B\rotate_around_atom_action\20250812_140135\rotate_around_atom_action_evaluation_wrongs_fixed.csv
Action: rotate_around_atom_action, use_exact_match: False
Rows: 247


Classifying:   0%|          | 0/247 [00:00<?, ?it/s]d:\AI\PythonProjects\AtomWorldBench\.conda\Lib\site-packages\pymatgen\io\cif.py:1313: UserWarning: Some occupancies ([2.0]) sum to > 1! If they are within the occupancy_tolerance, they will be rescaled. The current occupancy_tolerance is set to: 1.0
  if struct := self._get_structure(data, primitive, symmetrized, check_occu=check_occu):
d:\AI\PythonProjects\AtomWorldBench\src\utils\dataloader.py:33: UserWarning: Issues encountered while parsing CIF: Some occupancies ([2.0]) sum to > 1! If they are within the occupancy_tolerance, they will be rescaled. The current occupancy_tolerance is set to: 1.0
  structures = parser.parse_structures(primitive=primitive, check_occu=False)
Classifying:   8%|▊         | 20/247 [00:00<00:01, 182.29it/s]d:\AI\PythonProjects\AtomWorldBench\src\utils\dataloader.py:33: UserWarning: Issues encountered while parsing CIF: 8 fractional coordinates rounded to ideal values to avoid issues with finite precision.


...


Wrong type distribution:
wrong_type
StructureMismatch    116
CIFParsingError      104
OutputFormatError     16
AtomCountMismatch      3
Name: count, dtype: int64
Saved to ..\..\results\AtomWorld\llama3_70B\rotate_around_atom_action\20250813_135722\rotate_around_atom_action_evaluation_wrongs_fixed_relabeled.csv

Processing: ..\..\results\AtomWorld\llama3_70B\move_atom_action\20250813_135723\move_atom_action_evaluation_wrongs_fixed.csv
Action: move_atom_action, use_exact_match: False
Rows: 117


Classifying:   3%|▎         | 4/117 [00:00<00:08, 13.68it/s]d:\AI\PythonProjects\AtomWorldBench\src\utils\dataloader.py:33: UserWarning: Issues encountered while parsing CIF: 22 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  structures = parser.parse_structures(primitive=primitive, check_occu=False)
d:\AI\PythonProjects\AtomWorldBench\src\utils\dataloader.py:33: UserWarning: Issues encountered while parsing CIF: 3 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  structures = parser.parse_structures(primitive=primitive, check_occu=False)
Classifying:   9%|▊         | 10/117 [00:00<00:03, 28.88it/s]d:\AI\PythonProjects\AtomWorldBench\src\utils\dataloader.py:33: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  structures = parser.parse_structures(primitive=primitive, check_occu=False)
d:\AI\PythonProjects\AtomWorldBench


Wrong type distribution:
wrong_type
StructureMismatch    76
CIFParsingError      39
OutputFormatError     1
AtomCountMismatch     1
Name: count, dtype: int64
Saved to ..\..\results\AtomWorld\llama3_70B\move_atom_action\20250813_135723\move_atom_action_evaluation_wrongs_fixed_relabeled.csv

Processing: ..\..\results\AtomWorld\llama3_70B\move_towards_atom_action\20250813_135728\move_towards_atom_action_evaluation_wrongs_fixed.csv
Action: move_towards_atom_action, use_exact_match: False
Rows: 174


Classifying:   3%|▎         | 5/174 [00:00<00:04, 42.03it/s]d:\AI\PythonProjects\AtomWorldBench\src\utils\dataloader.py:33: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  structures = parser.parse_structures(primitive=primitive, check_occu=False)
d:\AI\PythonProjects\AtomWorldBench\src\utils\dataloader.py:33: UserWarning: Issues encountered while parsing CIF: 3 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  structures = parser.parse_structures(primitive=primitive, check_occu=False)
d:\AI\PythonProjects\AtomWorldBench\.conda\Lib\site-packages\pymatgen\io\cif.py:1313: UserWarning: Some occupancies ([2.0]) sum to > 1! If they are within the occupancy_tolerance, they will be rescaled. The current occupancy_tolerance is set to: 1.0
  if struct := self._get_structure(data, primitive, symmetrized, check_occu=check_occu):
d:\AI\PythonProjects\AtomWorldBench\src


Wrong type distribution:
wrong_type
StructureMismatch    148
CIFParsingError       20
OutputFormatError      5
AtomCountMismatch      1
Name: count, dtype: int64
Saved to ..\..\results\AtomWorld\llama3_70B\move_towards_atom_action\20250813_135728\move_towards_atom_action_evaluation_wrongs_fixed_relabeled.csv

